In [ ]:
import os
import json
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [ ]:
directory = "./Combined"
api_bookings = []

In [ ]:
for filename in os.listdir(directory):
    if filename.endswith(".json"):
        file_path = os.path.join(directory, filename)

        with open(file_path, 'r') as file:
            try:
                json_data = json.load(file)
                for index, item in enumerate(json_data):
                    api_bookings.append({
                        "listIndex": index,
                        "status": item.get("status"),
                        "pp_status": item.get("pp_status"),
                        "created_original_start": item.get("created_original_start"),
                        "created_original_end": item.get("created_original_end"),
                        "elapse_sec": item.get("elapse_sec"),
                        "hit_by_elapse": item.get("hit_by_elapse"),
                        "hits": item.get("hits"),
                        "bookings" : item.get("bookings")
                    })

            except json.JSONDecodeError:
                print(f"Error decoding JSON in file: {filename}")

In [ ]:
api_bookings_df = pd.DataFrame(api_bookings)
api_bookings_df.tail(5)

In [ ]:
bookings_data = []

In [ ]:
for bookings_dict in api_bookings_df['bookings']:
    if bookings_dict:
        for booking_id, booking_table in bookings_dict.items():
            bookings_data.append({
                "booking_id": booking_id,
                **booking_table
            })

In [ ]:
bookings_df_original_raw = pd.DataFrame(bookings_data)
bookings_df_original_raw.head(5)

In [ ]:
bookings_df_original_raw.shape

In [ ]:
# Ensure 'changed_original' is a datetime object for accurate comparisons
bookings_df_original_raw['changed_original'] = pd.to_datetime(bookings_df_original_raw['changed_original'],format='%Y-%m-%dT%H:%M:%S.%fZ', errors='coerce')

# Drop duplicates, keeping the row with the latest 'changed_original'
bookings_df_original = bookings_df_original_raw.sort_values(by=['booking_id', 'changed_original']).drop_duplicates(subset='booking_id', keep='last')


In [ ]:
bookings_df_original.shape

In [ ]:
## Reduce the columns null columns and unneccessary columns ## 205 columns dropped.
bookings_df = bookings_df_original.loc[:,['accommodation_first_start_original','accommodation_from','accommodation_last_end_date','accommodation_last_end_original','accommodation_until','activities_enddate','activities_startdate','activity_first_start_date','activity_last_end_date','agency_full_name','agency_id','agency_invoice_settings','all_end','all_start','booking_id','cancellation_date','changed_original','confirmed','confirmed_date','contact_id','created_original','school_id','school_name','course_from','course_last_end_date','course_last_end_original','course_period','course_until','first_course_start','insurance_first_start_date','insurance_end','insurance_from','insurance_last_end_date_original','amount','amount_initial','amount_open','amount_payed_original','amount_refund','amount_total_original','document_date_original','document_number','document_number_all','document_numbers_invoices','has_invoice_data','invoice_status','open_payment_at_school','open_payment_prior_to_arrival','payments','payments_local','service_from','service_until','customer_gender','customer_name','customer_nationality_original','customer_number','email','transfer_dates'
]]
bookings_df.shape

In [ ]:
column_order = [
     
 ## bookings  
    'booking_id',
    ## booking creation & statuses
    'created_original', 'confirmed', 'confirmed_date', 'changed_original',
    'cancellation_date', 
 ## school  
    'school_id', 'school_name', 
 ## student (customer)
    'contact_id', 'customer_name', 'customer_number', 'customer_gender', 
    'customer_nationality_original', 'email', 
 ## agency
    'agency_id', 'agency_full_name', 'agency_invoice_settings', 
 ## Booking Details
    ## Booking Dates (ALL)
        'all_start', 'all_end', 
        ## Course
        'first_course_start', 'course_from', 'course_until', 'course_last_end_date', 
        'course_last_end_original', 'course_period', 
        
        ## Service
        'service_from', 'service_until', 
    
        ## Accommodation
        'accommodation_first_start_original', 'accommodation_from', 'accommodation_last_end_date', 
        'accommodation_last_end_original', 'accommodation_until', 
        
        ## Activities
        'activities_startdate', 
        'activities_enddate', 'activity_first_start_date', 'activity_last_end_date', 
    
        ## Insurance
        'insurance_first_start_date', 'insurance_end', 'insurance_from', 
        'insurance_last_end_date_original', 
        ## Transfers
        'transfer_dates',
    ## Document (all)
        'document_number', 'document_date_original',  'document_number_all',
        'document_numbers_invoices', 'has_invoice_data', 'invoice_status',
    ## Document Amounts
    'amount', 'amount_total_original', 'amount_initial', 
    ## Open amount
    'amount_open','open_payment_at_school', 'open_payment_prior_to_arrival',
    ## Refund
    'amount_refund', 
    ## Payment
    'amount_payed_original', 'payments_local', 'payments'
]
bookings_df = bookings_df[column_order]

In [ ]:
student_columns = [
    'contact_id', 'customer_name', 'customer_number', 'customer_gender', 
    'customer_nationality_original', 'email'
]

In [ ]:
school_columns = [
    'school_id', 'school_name'
]

In [ ]:
document_columns = [
        'document_number', 'document_date_original',  'document_number_all',
        'document_numbers_invoices', 'has_invoice_data', 'invoice_status',
            'amount', 'amount_total_original', 'amount_initial', 
            'amount_open','open_payment_at_school', 'open_payment_prior_to_arrival',
            'amount_refund', 
            'amount_payed_original', 'payments_local', 'payments'
]

In [ ]:
list_date_columns = [
    'course_from', 'course_until','course_last_end_original',
    'insurance_from','insurance_end','insurance_last_end_date_original',
    'service_from', 'service_until', 
    'accommodation_from', 'accommodation_until','accommodation_last_end_original',
    'activities_startdate','activities_enddate',
    'transfer_dates'
]

In [ ]:
date_columns = [
    'all_start', 'all_end','document_date_original',
    'confirmed_date', 'created_original',
    ##'changed_original',
    'cancellation_date',
    'first_course_start','course_last_end_date', 
    'insurance_first_start_date',
    'accommodation_first_start_original','accommodation_last_end_date',
    'activity_first_start_date','activity_last_end_date'
]

In [ ]:
bookings_df = bookings_df.dropna(subset=['all_start', 'all_end'])
print(bookings_df[['all_start', 'all_end']].isna().sum())

In [ ]:
for col in date_columns:
     bookings_df[col] = pd.to_datetime(bookings_df[col], format='%Y-%m-%dT%H:%M:%S.%fZ', errors='coerce')

In [ ]:
print(bookings_df[['all_start', 'all_end']].dtypes)

In [ ]:
def handle_list_date_columns(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: [] if x is None else ([x] if isinstance(x, str) else x))
        df[col] = df[col].apply(
            lambda lst: [pd.to_datetime(date,format='%Y-%m-%dT%H:%M:%S.%fZ', errors='coerce').date() for date in lst]
            if isinstance(lst, list) else lst
        )
    return df

In [ ]:
bookings_df = handle_list_date_columns(bookings_df, list_date_columns)

In [ ]:
def handle_list_date_columns2(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: [] if x is None else ([x] if isinstance(x, str) else x))
        df[col] = df[col].apply(
            lambda lst: [pd.to_datetime(date, format='%Y-%m-%dT%H:%M:%S.%fZ', errors='coerce').date() 
                         for date in lst] if isinstance(lst, list) else lst
        )
        df = df.explode(col)
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

In [ ]:
print(bookings_df[['course_from', 'course_until']].dtypes)

In [ ]:
bookings_df.loc[:, 'days_diff'] = (bookings_df['all_end'] - bookings_df['all_start']).dt.days
bookings_df.loc[:, 'weeks_diff'] = (bookings_df['all_end'] - bookings_df['all_start']).dt.days // 7

In [ ]:
bookings_df.loc[:,['booking_id'] + date_columns + list_date_columns + ['days_diff'] + ['weeks_diff']]

In [ ]:
pd.options.mode.copy_on_write = True
def get_year_week_numbers(start_date, end_date):
    current_date = start_date
    week_numbers = []
    while current_date <= end_date:
        year_week = f"{current_date.year}-{current_date.isocalendar()[1]}"
        week_numbers.append(year_week)
        current_date += pd.Timedelta(days=7)
    return sorted(week_numbers, key=lambda x: (int(x.split('-')[0]), int(x.split('-')[1])))

def get_week_numbers(start_date, end_date):
    current_date = start_date
    week_numbers = []
    while current_date <= end_date:
        week = current_date.isocalendar()[1]
        week_numbers.append(week)
        current_date += pd.Timedelta(days=7)
    week_numbers_sorted = sorted(set(week_numbers))
    return week_numbers_sorted

def get_adjusted_week_numbers(start_date, end_date):
    current_date = start_date
    week_numbers = []
    first_week = start_date.isocalendar()[1]
    while current_date <= end_date:
        week = current_date.isocalendar()[1]
        if week < first_week: 
            week += 52
        week_numbers.append(week)
        current_date += pd.Timedelta(days=7)
    return sorted(set(week_numbers))

In [ ]:
bookings_df = bookings_df.copy()
bookings_df['week_numbers'] = bookings_df.apply(
    lambda row: get_adjusted_week_numbers(row['all_start'], row['all_end']), axis=1
)
bookings_df['year_week_numbers'] = bookings_df.apply(
    lambda row: get_year_week_numbers(row['all_start'], row['all_end']), axis=1
)

In [ ]:
print(bookings_df[['all_start', 'all_end', 'week_numbers', 'year_week_numbers']].head())

In [ ]:
##bookings_df.to_excel('bookings_df.xlsx', index=False)

In [ ]:
bookings_df['total_payment'] = bookings_df['amount_payed_original'] + bookings_df['amount_refund']


In [ ]:
amount_columns2 = [ 
    'amount','total_payment','amount_open'
    
]

In [ ]:
bookings_df.loc[:,['booking_id']+['customer_name']+['customer_number']+amount_columns2].head()

In [ ]:
bookings_df.shape

In [ ]:
bookings_df = bookings_df[bookings_df['amount'] != 0]

In [ ]:
bookings_df.shape

In [ ]:
column_order = [
     
 ## bookings  
    'booking_id',
    ## booking creation & statuses
    'created_original', 'confirmed_date', 'changed_original',
    'cancellation_date', 
 ## school  
                                                                                            ##'school_id', 
    'school_name', 
 ## student (customer)
                                                                                            ##'contact_id', 
    'customer_name', 'customer_number', 
                                                                                        ##'customer_gender', 
                                                                                        ##'customer_nationality_original', 'email', 
 ## agency
                                                                                         ##   'agency_id', 'agency_full_name', 'agency_invoice_settings', 
 ## Booking Details
    ## Booking Dates (ALL)
        'all_start', 'all_end', 
        ## Course
        'course_from', 'course_until',
                                                                                        ##'course_period', 
    ## Document (all)
        'document_number', 'document_date_original',
        'document_numbers_invoices', 'has_invoice_data',
    ## Amounts
    'amount','total_payment','amount_open',
    ## Weeks
    'week_numbers', 'year_week_numbers'
    
]
bookings_df_selected = bookings_df[column_order]

In [ ]:
bookings_df_selected

In [ ]:
bookings_df_selected.dtypes

In [ ]:
column_order = [    
    'booking_id', ## object -> make it integer
    'created_original', ## datetime64[ns]
    'confirmed_date', ## datetime64[ns]
    'changed_original',## datetime64[ns]
    'cancellation_date', ## datetime64[ns]
    'school_name', ## object -> make it string
    'customer_name', ## object -> make it string
    'customer_number', ## object -> make it string 
    'course_from', ## object list of dates
    'course_until', ## object list of dates
    'document_number', ## string
    'document_date_original', #  datetime64[ns]
    'document_numbers_invoices', ## string
    'has_invoice_data',  ## 0 or 1
    'amount', ## float64
    'total_payment', ## float64
    'amount_open', ## float64
    'all_start', ## datetime64[ns]
    'all_end', ## datetime64[ns]
    'week_numbers', ## object (list of integers) -> make it max string
    'year_week_numbers' ## object (list of strings) -> make it max string
    
]
bookings_df_selected = bookings_df[column_order]

In [ ]:
def transform_dataframe(df):
    required_columns = [
        'booking_id', 'created_original', 'confirmed_date', 'changed_original', 'cancellation_date',
        'school_name', 'customer_name', 'customer_number', 'course_from', 'course_until',
        'document_number', 'document_date_original', 'document_numbers_invoices', 'has_invoice_data',
        'amount', 'total_payment', 'amount_open', 'all_start', 'all_end',
        'week_numbers', 'year_week_numbers'
    ]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    # Convert data types
    df['booking_id'] = df['booking_id'].astype(int)  # object -> integer
    df['created_original'] = pd.to_datetime(df['created_original'], errors='coerce')  # datetime
    df['confirmed_date'] = pd.to_datetime(df['confirmed_date'], errors='coerce')  # datetime
    df['changed_original'] = pd.to_datetime(df['changed_original'], errors='coerce')  # datetime
    df['cancellation_date'] = pd.to_datetime(df['cancellation_date'], errors='coerce')  # datetime
    df['school_name'] = df['school_name'].astype(str)  # object -> string
    df['customer_name'] = df['customer_name'].astype(str)  # object -> string
    df['customer_number'] = df['customer_number'].astype(str)  # object -> string
    df['document_number'] = df['document_number'].astype(str)  # object -> string
    df['document_date_original'] = pd.to_datetime(df['document_date_original'], errors='coerce')  # datetime
    df['document_numbers_invoices'] = df['document_numbers_invoices'].astype(str)  # object -> string
    df['has_invoice_data'] = df['has_invoice_data'].astype(int)  # boolean-like to integer
    df['amount'] = df['amount'].astype(float)  # float64
    df['total_payment'] = df['total_payment'].astype(float)  # float64
    df['amount_open'] = df['amount_open'].astype(float)  # float64
    df['all_start'] = pd.to_datetime(df['all_start'], errors='coerce')  # datetime
    df['all_end'] = pd.to_datetime(df['all_end'], errors='coerce')  # datetime

    # Transform object lists to strings
    df['course_from'] = df['course_from'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x))
    df['course_until'] = df['course_until'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x))
    df['week_numbers'] = df['week_numbers'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x))
    df['year_week_numbers'] = df['year_week_numbers'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x))
    
    return df

bookings_df_selected = transform_dataframe(bookings_df_selected)

print(bookings_df_selected.dtypes)


In [ ]:
# Database connection parameters
server = os.environ.get("SQL_SERVER", "localhost\\SQLEXPRESS")
database = "booking_db"
driver = "ODBC Driver 17 for SQL Server"

connection_string = f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
engine = create_engine(connection_string)
table_name = "bookings" 
try:
    # If the table already exists, replace it (change 'replace' to 'append' to keep existing data)
    bookings_df_selected.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Data exported successfully to {table_name} table.")
except Exception as e:
    print("Error exporting data:", e)